# fractalsearch in Colab

Run the real repository manually, watch it in the original dashboard, then give the same folder to a coding agent.

### 1. Get the project

This is a normal Git clone. Every source file stays visible and editable. Re-running this cell keeps work already in the Colab session.

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/johnny0595/fractal-autoresearch-colab.git"
PROJECT = Path("/content/fractal-autoresearch-colab")
fresh_clone = not PROJECT.exists()
if fresh_clone:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)

# Keep experiments and Git history local to this temporary Colab runtime.
subprocess.run(["git", "config", "user.name", "Fractalsearch Researcher"], check=True)
subprocess.run(["git", "config", "user.email", "research@local.invalid"], check=True)
if fresh_clone:
    Path("runs.jsonl").write_text("")
    subprocess.run(["git", "add", "runs.jsonl"], check=True)
    subprocess.run(["git", "commit", "-m", "Start local research log"], check=True)
if "origin" in subprocess.check_output(["git", "remote"], text=True).split():
    subprocess.run(["git", "remote", "remove", "origin"], check=True)

print(PROJECT)

### 2. Install the small web stack

Colab already provides PyTorch, NumPy, and the GPU runtime.

In [ ]:
%pip -q install "pillow>=11" "fastapi>=0.115" "uvicorn>=0.32"

In [ ]:
import torch
assert torch.cuda.is_available(), "Choose Runtime → Change runtime type → GPU, then reconnect."
print("GPU:", torch.cuda.get_device_name(0))

### 3. Read the research brief

`AGENT.md` defines the objective, the files that are off-limits, and the experiment loop. The agent will read this same file later.

In [ ]:
print(Path("AGENT.md").read_text())

### 4. Read the baseline

A solution only needs to implement `fit(...)` and `predict(...)`. This plain coordinate MLP is the starting point.

In [ ]:
print(Path("solutions/baseline_mlp.py").read_text())

### 5. Run one experiment yourself

The evaluator owns the data, timer, metric, and saved artifacts. Thirty seconds is enough to verify the workflow; autonomous runs use the original five-minute budget.

In [ ]:
BUDGET_SECONDS = 30
subprocess.run([
    "python", "-m", "harness.evaluate",
    "solutions/baseline_mlp.py", "--budget", str(BUDGET_SECONDS),
], check=True)

In [ ]:
import json
runs = [json.loads(line) for line in Path("runs.jsonl").read_text().splitlines() if line]
runs[-1]

### 6. Open the original dashboard

It reads `runs.jsonl` and updates through the original live event stream whenever a run finishes.

In [ ]:
import socket, sys, time

def port_is_open(port):
    with socket.socket() as sock:
        return sock.connect_ex(("127.0.0.1", port)) == 0

if not port_is_open(8000):
    dashboard_log = open("/tmp/fractalsearch-dashboard.log", "w")
    dashboard_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "dashboard.app:app",
         "--host", "0.0.0.0", "--port", "8000"],
        stdout=dashboard_log, stderr=subprocess.STDOUT,
    )
    for _ in range(40):
        if port_is_open(8000):
            break
        time.sleep(0.25)
    else:
        raise RuntimeError(Path("/tmp/fractalsearch-dashboard.log").read_text())

from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=850)

## Run the research loop with an agent

Open **Tools → Terminal** in Colab and stay in `/content/fractal-autoresearch-colab`. The terminal and notebook share the same files and GPU. Keep the dashboard cell above open; each completed run appears automatically.

Pick one agent. Authentication happens in the terminal—do not paste API keys into this notebook.

### Codex CLI

```bash
cd /content/fractal-autoresearch-colab
curl -fsSL https://chatgpt.com/codex/install.sh | sh
export PATH="/root/.local/bin:$PATH"
codex login --device-auth
codex --sandbox workspace-write --approve-for-me \
  "Read AGENTS.md. The dashboard is already running. Verify PyTorch can access CUDA. If CUDA or Git is blocked, rerun only that command with elevated permission. Then establish the baseline and start the research loop. Keep going until I interrupt you."
```

### Claude Code

```bash
cd /content/fractal-autoresearch-colab
curl -fsSL https://claude.ai/install.sh | bash
export PATH="/root/.local/bin:$PATH"
claude --permission-mode acceptEdits \
  "Read CLAUDE.md. Establish the baseline, then start the research loop. Keep going until I interrupt you."
```

Use **Ctrl-C** when you want the loop to stop. The original protocol is intentionally open-ended.